In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
from IPython.display import HTML

In [2]:
def show_fig(fig):
    return HTML(fig.to_html(full_html=False, include_plotlyjs='cdn'))

region_mapping={'Africa (EI)':'África','Asia Pacific (EI)':'Asia Pacífico','Europe (EI)':'Europa','Middle East (EI)':'Medio Oriente','North America (EI)':'Norteamérica','South and Central America (EI)':'Sudamérica y Centroamérica'}
regiones=['Africa','Asia','Europe','North America','Oceania','South America','World']
tr={'Africa':'África','Asia':'Asia','Europe':'Europa','North America':'Norteamérica','Oceania':'Oceanía','South America':'Sudamérica','World':'Mundo'}
def parse_a2(path='../data/A_A2_r_230822.081459.xlsx'):
    r=pd.read_excel(path,sheet_name='A2',skiprows=5)
    yc=[c for c in r.columns if isinstance(c,(int,float))]
    r=r[r['Region and fuel'].notna()].copy()
    r['Region and fuel']=r['Region and fuel'].astype(str)
    r=r[~r['Region and fuel'].str.startswith('Data source:')].copy()
    rows=[]; cur=None
    for _,row in r.iterrows():
        lab=row['Region and fuel'].strip(); vals=row[yc]
        if vals.isna().all(): cur=lab; continue
        rows.append({'region':cur,'fuel':lab,**{int(y):row[y] for y in yc}})
    d=pd.DataFrame(rows).melt(id_vars=['region','fuel'],var_name='year',value_name='consumo_quad_btu')
    d['consumo_kwh']=d['consumo_quad_btu']*2.9307107e11
    d['fuel_es']=d['fuel'].replace({'Liquid fuels':'Combustibles líquidos','Natural gas':'Gas natural','Coal':'Carbón','Nuclear':'Nuclear','Other':'Otras','Total':'Total'})
    d['region_es']=d['region'].replace({'Americas':'Américas','Europe and Eurasia':'Europa y Eurasia','Asia Pacific':'Asia Pacífico','Africa and Middle East':'África y Medio Oriente','World':'Mundo'})
    return d
def add_split(fig,x=2026):
    fig.add_vline(x=x,line_dash='dash',line_color='black')
    fig.add_annotation(x=x-0.5,y=1.04,xref='x',yref='paper',text='Histórico',showarrow=False,xanchor='right')
    fig.add_annotation(x=x+0.5,y=1.04,xref='x',yref='paper',text='Proyección',showarrow=False,xanchor='left')

In [3]:
#| label: fig-panorama-fuentes-abs
d=pd.read_csv('../data/3960AFa8.csv',skiprows=3)
cols=[c for c in d.columns if c not in ['Año','Unidades']]
fig=px.area(d,x='Año',y=cols,labels={'value':'Energía primaria [TJ]','variable':'Fuente'})
fig.update_layout(hovermode='x unified')
show_fig(fig)

In [4]:
#| label: fig-panorama-fuentes-pct
d=pd.read_csv('../data/3960AFa8.csv',skiprows=3)
cols=[c for c in d.columns if c not in ['Año','Unidades']]
fig=px.area(d,x='Año',y=cols,groupnorm='percent',labels={'value':'%','variable':'Fuente'})
fig.update_layout(hovermode='x unified')
show_fig(fig)

In [5]:
#| label: fig-primaria-region-abs
d=pd.read_csv('../data/primary-energy-consumption-by-region.csv')
d['Entity']=d['Entity'].replace(region_mapping)
fig=px.area(d,x='Year',y='Primary energy consumption',color='Entity',labels={'Year':'Año','Primary energy consumption':'Energía primaria [TJ]','Entity':'Región'})
fig.update_layout(hovermode='x unified')
show_fig(fig)

In [6]:
#| label: fig-primaria-region-pct
d=pd.read_csv('../data/primary-energy-consumption-by-region.csv')
d['Entity']=d['Entity'].replace(region_mapping)
fig=px.area(d,x='Year',y='Primary energy consumption',color='Entity',groupnorm='percent',labels={'Year':'Año','Primary energy consumption':'%','Entity':'Región'})
fig.update_layout(hovermode='x unified')
show_fig(fig)

In [7]:
#| label: fig-renovables-escenarios
d=pd.read_csv('../data/iea_renovables_mundo_capacidad_es.csv')
s=d[d['producto']!='Total'].copy(); m=s[s['escenario']=='Caso principal'].copy(); a=s[s['escenario']=='Caso acelerado'].copy(); t=d[d['escenario']=='Meta'].copy()
prods=m['producto'].drop_duplicates().tolist(); colors=px.colors.qualitative.Plotly; cmap={p:colors[i%len(colors)] for i,p in enumerate(prods)}
fig=go.Figure()
for p in prods:
    mp=m[m['producto']==p].sort_values('anio'); ap=a[a['producto']==p].sort_values('anio')
    fig.add_trace(go.Bar(x=mp['anio'],y=mp['valor'],name=p,legendgroup=p,offsetgroup='Caso principal',marker={'color':cmap[p]}))
    fig.add_trace(go.Bar(x=ap['anio'],y=ap['valor'],name=p,legendgroup=p,offsetgroup='Caso acelerado',showlegend=False,marker={'color':cmap[p],'pattern':{'shape':'/'}}))
fig.add_hline(y=11500,line_dash='dash',line_color='black',annotation_text='Meta COP28: triplicar renovables para 2030',annotation_position='top left')
fig.add_trace(go.Scatter(x=t['anio'],y=t['valor'],mode='markers+text',text=['Ambición renovable actual 2030'],textposition='top center',marker={'size':10,'symbol':'diamond','color':'red'}))
fig.update_layout(barmode='stack',hovermode='x unified',xaxis_title='Año',yaxis_title='Capacidad instalada [GW]',legend_title='Tecnología')
show_fig(fig)

In [8]:
#| label: fig-oil-percap-region
d=pd.read_csv('https://ourworldindata.org/grapher/per-capita-oil.csv?v=1&csvType=full&useColumnShortNames=true',storage_options={'User-Agent':'Our World In Data data fetch/1.0'})
d=d[d['entity'].isin(regiones)].copy(); d['entity']=d['entity'].replace(tr)
fig=px.line(d,x='year',y='oil_per_capita__kwh',color='entity',labels={'year':'Año','oil_per_capita__kwh':'Consumo de petróleo [kWh per cápita]','entity':'Región'})
fig.update_layout(hovermode='x unified')
show_fig(fig)

In [9]:
#| label: fig-elec-fuente-abs
d=pd.read_csv('https://ourworldindata.org/grapher/electricity-prod-source-stacked.csv?v=1&csvType=full&useColumnShortNames=true',storage_options={'User-Agent':'Our World In Data data fetch/1.0'})
w=d[d['entity']=='World'].copy(); cm={'other_renewables_excluding_bioenergy_generation__twh_chart_electricity_prod_source_stacked':'Otras renovables (sin bioenergía)','bioenergy_generation__twh_chart_electricity_prod_source_stacked':'Bioenergía','solar_generation__twh_chart_electricity_prod_source_stacked':'Solar','wind_generation__twh_chart_electricity_prod_source_stacked':'Eólica','hydro_generation__twh_chart_electricity_prod_source_stacked':'Hidroeléctrica','nuclear_generation__twh_chart_electricity_prod_source_stacked':'Nuclear','oil_generation__twh_chart_electricity_prod_source_stacked':'Petróleo','gas_generation__twh_chart_electricity_prod_source_stacked':'Gas','coal_generation__twh_chart_electricity_prod_source_stacked':'Carbón'}
sc=[c for c in w.columns if c in cm]; w=w.rename(columns=cm); src=[cm[c] for c in sc]
fig=px.area(w,x='year',y=src,labels={'year':'Año','value':'Generación eléctrica [TWh]','variable':'Fuente'})
fig.update_layout(hovermode='x unified')
show_fig(fig)

In [10]:
#| label: fig-elec-fuente-pct
d=pd.read_csv('https://ourworldindata.org/grapher/electricity-prod-source-stacked.csv?v=1&csvType=full&useColumnShortNames=true',storage_options={'User-Agent':'Our World In Data data fetch/1.0'})
w=d[d['entity']=='World'].copy(); cm={'other_renewables_excluding_bioenergy_generation__twh_chart_electricity_prod_source_stacked':'Otras renovables (sin bioenergía)','bioenergy_generation__twh_chart_electricity_prod_source_stacked':'Bioenergía','solar_generation__twh_chart_electricity_prod_source_stacked':'Solar','wind_generation__twh_chart_electricity_prod_source_stacked':'Eólica','hydro_generation__twh_chart_electricity_prod_source_stacked':'Hidroeléctrica','nuclear_generation__twh_chart_electricity_prod_source_stacked':'Nuclear','oil_generation__twh_chart_electricity_prod_source_stacked':'Petróleo','gas_generation__twh_chart_electricity_prod_source_stacked':'Gas','coal_generation__twh_chart_electricity_prod_source_stacked':'Carbón'}
sc=[c for c in w.columns if c in cm]; w=w.rename(columns=cm); src=[cm[c] for c in sc]
fig=px.area(w,x='year',y=src,groupnorm='percent',labels={'year':'Año','value':'%','variable':'Fuente'})
fig.update_layout(hovermode='x unified')
show_fig(fig)

In [11]:
#| label: fig-elec-percap-region-abs
d=pd.read_csv('https://ourworldindata.org/grapher/per-capita-electricity-generation.csv?v=1&csvType=full&useColumnShortNames=true',storage_options={'User-Agent':'Our World In Data data fetch/1.0'})
d=d[d['entity'].isin(regiones)].copy(); d=d[d['year']<=2023].copy(); d['region']=d['entity'].replace(tr)
v='per_capita_electricity_generation__kwh'
if v not in d.columns:
  cs=[c for c in d.columns if c not in ['entity','code','year']]; v=next((c for c in cs if 'per_capita' in c),cs[0])
fig=px.area(d.sort_values(['region','year']),x='year',y=v,color='region',labels={'year':'Año',v:'Generación eléctrica per cápita [kWh]','region':'Región'})
fig.update_layout(hovermode='x unified')
show_fig(fig)

In [12]:
#| label: fig-elec-percap-region-pct
d=pd.read_csv('https://ourworldindata.org/grapher/per-capita-electricity-generation.csv?v=1&csvType=full&useColumnShortNames=true',storage_options={'User-Agent':'Our World In Data data fetch/1.0'})
d=d[d['entity'].isin(regiones)].copy(); d=d[d['year']<=2023].copy(); d['region']=d['entity'].replace(tr)
v='per_capita_electricity_generation__kwh'
if v not in d.columns:
  cs=[c for c in d.columns if c not in ['entity','code','year']]; v=next((c for c in cs if 'per_capita' in c),cs[0])
fig=px.area(d.sort_values(['region','year']),x='year',y=v,color='region',groupnorm='percent',labels={'year':'Año',v:'%','region':'Región'})
fig.update_layout(hovermode='x unified')
show_fig(fig)

In [13]:
#| label: fig-primaria-2050-fuente-abs
dl=parse_a2(); d=dl[(dl['region']=='World')&(dl['fuel']!='Total')].copy()
fig=px.area(d.sort_values(['fuel_es','year']),x='year',y='consumo_kwh',color='fuel_es',labels={'year':'Año','consumo_kwh':'Consumo de energía primaria [kWh]','fuel_es':'Fuente'})
add_split(fig); fig.update_layout(hovermode='x unified')
show_fig(fig)

In [14]:
#| label: fig-primaria-2050-fuente-pct
dl=parse_a2(); d=dl[(dl['region']=='World')&(dl['fuel']!='Total')].copy()
fig=px.area(d.sort_values(['fuel_es','year']),x='year',y='consumo_kwh',color='fuel_es',groupnorm='percent',labels={'year':'Año','consumo_kwh':'%','fuel_es':'Fuente'})
add_split(fig); fig.update_layout(hovermode='x unified')
show_fig(fig)

In [15]:
#| label: fig-primaria-2050-region-abs
dl=parse_a2(); d=dl[(dl['fuel']=='Total')&(dl['region']!='World')].copy()
fig=px.area(d.sort_values(['region_es','year']),x='year',y='consumo_kwh',color='region_es',labels={'year':'Año','consumo_kwh':'Consumo de energía primaria [kWh]','region_es':'Región'})
add_split(fig); fig.update_layout(hovermode='x unified')
show_fig(fig)

In [16]:
#| label: fig-primaria-2050-region-pct
dl=parse_a2(); d=dl[(dl['fuel']=='Total')&(dl['region']!='World')].copy()
fig=px.area(d.sort_values(['region_es','year']),x='year',y='consumo_kwh',color='region_es',groupnorm='percent',labels={'year':'Año','consumo_kwh':'%','region_es':'Región'})
add_split(fig); fig.update_layout(hovermode='x unified')
show_fig(fig)

In [17]:
#| label: fig-geotermia-mapa
dg=pd.read_csv('../data/merged_geothermal.csv')
dm=dg[['Country','2024 [1]','2025 [2]','2025 [3]']].copy(); dm.columns=['País','2024 [1]','2025 [2]','2025 [3]']
dl=dm.melt(id_vars=['País'],var_name='Año',value_name='Capacidad [MW]')
fig=px.choropleth(dl,locations='País',locationmode='country names',color='Capacidad [MW]',hover_name='País',animation_frame='Año',color_continuous_scale=px.colors.sequential.Plasma)
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration']=1500
fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration']=500
show_fig(fig)

/tmp/ipykernel_21633/238512454.py:5: DeprecationWarning:

The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.



In [18]:
#| label: tbl-geotermia-top10
#| tbl-cap: "Top 10 países con mayor capacidad instalada geotérmica"
dg=pd.read_csv('../data/merged_geothermal.csv')
a=dg[['Country','2024 [1]']].sort_values(by='2024 [1]',ascending=False).head(10).reset_index(drop=True)
b=dg[['Country','2025 [2]']].sort_values(by='2025 [2]',ascending=False).head(10).reset_index(drop=True)
c=dg[['Country','2025 [3]']].sort_values(by='2025 [3]',ascending=False).head(10).reset_index(drop=True)
top_10=pd.concat([a,b,c],axis=1)
top_10.columns=['País','2024 [1]','País','2025 [2]','País','2025 [3]']
top_10

,País,2024 [1],País,2025 [2],País,2025 [3]
0,United States,2725.000,United States,3733.50,United States,3953.0
1,Indonesia,2638.800,Indonesia,2431.90,Indonesia,2742.0
2,Philippines,1951.800,Philippines,1937.00,Philippines,2034.0
3,Türkiye,1734.338,Türkiye,1726.11,Türkiye,1797.0
4,New Zealand,1275.400,New Zealand,1376.70,New Zealand,1259.0
5,Mexico,998.500,Mexico,941.00,Kenya,980.0
6,Kenya,939.730,Italy,834.00,Mexico,976.0
7,Iceland,787.580,Kenya,816.50,Italy,916.0
8,Italy,771.790,Iceland,779.40,Iceland,808.0
9,Japan,489.054,Japan,618.20,Japan,607.0
